# 📦 Optimisation Multi-Étages de la Valeur de Stock
## Modélisation MILP — Python + PuLP

**Objectif :** Minimiser la valeur totale du stock (MP + SF + PF) pour atteindre une cible de fin d'année fiscale (10 000 000 $) en fin septembre.

**Architecture :**
- **Partie 1 :** Paramètres généraux & horizon temporel
- **Partie 2 :** Données fictives réalistes (MP / SF / PF)
- **Partie 3 :** Calibration EOQ (stocks de sécurité)
- **Partie 4 :** Modélisation MILP complète (variables + fonction objectif + contraintes C1 → C13)

- **Partie 5 :** Résolution (appel solveur CBC + extraction des résultats + graphiques)

> ✅ Ce notebook contient **la modélisation ET la résolution** : partie 1 à 4 identiques au notebook de modélisation, partie 5 ajoutée ci-dessous, en suivant la même organisation (Paramètres → Variables → Fonction objectif → Contraintes → **Résolution** → **Affichage**).


## 0. Imports

In [1]:
# Installation si nécessaire :
# pip install pulp pandas numpy

import pulp
import pandas as pd
import numpy as np

print(f"PuLP version   : {pulp.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")


PuLP version   : 3.3.2
NumPy version  : 2.4.4
Pandas version : 3.0.2


## 1. Paramètres généraux

| Période | Index t |
|---------|---------|
| Juin    | 1       |
| Juillet | 2       |
| Août    | 3       |
| Septembre | 4     |


In [2]:
# --- Horizon temporel ---
periodes      = [1, 2, 3, 4]
noms_periodes = {1: "Juin", 2: "Juillet", 3: "Août", 4: "Septembre"}
T             = 4       # Fin d'horizon = fin septembre
t_juillet     = 2       # Dernier mois avant fermeture fournisseur
t_aout        = [3]     # Période(s) de fermeture fournisseur

# --- Cible fiscale ---
TARGET = 10_000_000     # Valeur de stock cible fin septembre ($)

# --- Paramètres économiques ---
tau     = 0.20          # Taux de possession annuel (20%)
z_alpha = 1.65          # Coefficient niveau de service 95%

# --- Big-M pour linéarisation des contraintes MOQ/MLS ---
M = 1_000_000

print("✅ Paramètres généraux définis")
print(f"   Horizon        : {list(noms_periodes.values())}")
print(f"   Cible fiscale  : {TARGET:,.0f} $")
print(f"   Taux possession: {tau*100:.0f}%")
print(f"   Niveau service : {z_alpha} (α = 95%)")


✅ Paramètres généraux définis
   Horizon        : ['Juin', 'Juillet', 'Août', 'Septembre']
   Cible fiscale  : 10,000,000 $
   Taux possession: 20%
   Niveau service : 1.65 (α = 95%)


## 2. Données fictives réalistes

### 2.1 Matières Premières (MP)

Paramètres par référence MP :
- `prix` : valeur unitaire ($)
- `stock_init` : stock réel à t=0 (extrait SAP)
- `LT` : lead time fournisseur (mois)
- `MOQ` : quantité minimale de commande
- `sigma_D` : écart-type de la demande (pour calcul SS)
- `C_passation` : coût fixe de passation de commande ($)


In [3]:
donnees_MP = {
    #  id     prix   stock_init   LT   MOQ   sigma_D   C_passation
    "MP1": {"prix": 50,   "stock_init": 5000,  "LT": 2, "MOQ": 500,  "sigma_D": 200, "C_passation": 300},
    "MP2": {"prix": 30,   "stock_init": 8000,  "LT": 1, "MOQ": 1000, "sigma_D": 150, "C_passation": 200},
    "MP3": {"prix": 120,  "stock_init": 2000,  "LT": 2, "MOQ": 200,  "sigma_D": 80,  "C_passation": 500},
    "MP4": {"prix": 15,   "stock_init": 12000, "LT": 1, "MOQ": 2000, "sigma_D": 300, "C_passation": 100},
    "MP5": {"prix": 200,  "stock_init": 1000,  "LT": 3, "MOQ": 100,  "sigma_D": 50,  "C_passation": 800},
}

# Affichage synthétique
df_MP = pd.DataFrame(donnees_MP).T
df_MP.index.name = "Référence"
print("Matières Premières :")
display(df_MP)


Matières Premières :


,prix,stock_init,LT,MOQ,sigma_D,C_passation
Référence,,,,,,
MP1,50,5000,2,500,200,300
MP2,30,8000,1,1000,150,200
MP3,120,2000,2,200,80,500
MP4,15,12000,1,2000,300,100
MP5,200,1000,3,100,50,800


### 2.2 Semi-Finis (SF)

Paramètres supplémentaires par rapport aux MP :
- `ct` : temps de cycle de production (mois)
- `MLS` : taille de lot minimale de production
- `charge` : charge unitaire (heures/unité produite)
- `C_lancement` : coût fixe de lancement d'un OF ($)


In [4]:
donnees_SF = {
    #  id     prix   stock_init   ct   MLS   sigma_D   C_lancement   charge
    "SF1": {"prix": 300,  "stock_init": 800,  "ct": 1, "MLS": 100, "sigma_D": 60, "C_lancement": 400, "charge": 2.0},
    "SF2": {"prix": 180,  "stock_init": 1200, "ct": 1, "MLS": 200, "sigma_D": 90, "C_lancement": 300, "charge": 1.5},
    "SF3": {"prix": 450,  "stock_init": 400,  "ct": 2, "MLS": 50,  "sigma_D": 30, "C_lancement": 600, "charge": 3.0},
}

df_SF = pd.DataFrame(donnees_SF).T
df_SF.index.name = "Référence"
print("Semi-Finis :")
display(df_SF)


Semi-Finis :


,prix,stock_init,ct,MLS,sigma_D,C_lancement,charge
Référence,,,,,,,
SF1,300.0,800.0,1.0,100.0,60.0,400.0,2.0
SF2,180.0,1200.0,1.0,200.0,90.0,300.0,1.5
SF3,450.0,400.0,2.0,50.0,30.0,600.0,3.0


### 2.3 Produits Finis (PF)

Paramètre supplémentaire : `demande` = demande client par période (seul étage avec demande externe réelle).


In [5]:
donnees_PF = {
    "PF1": {
        "prix": 800,  "stock_init": 300, "ct": 1, "MLS": 50,
        "demande": {1: 200, 2: 220, 3: 210, 4: 230},
        "sigma_D": 40, "C_lancement": 700, "charge": 4.0
    },
    "PF2": {
        "prix": 1200, "stock_init": 150, "ct": 1, "MLS": 30,
        "demande": {1: 100, 2: 110, 3: 105, 4: 120},
        "sigma_D": 25, "C_lancement": 900, "charge": 6.0
    },
}

df_PF = pd.DataFrame({l: {k: v for k, v in p.items() if k != "demande"}
                       for l, p in donnees_PF.items()}).T
df_PF.index.name = "Référence"
print("Produits Finis :")
display(df_PF)

print("\nDemande client par période :")
df_dem = pd.DataFrame({l: donnees_PF[l]["demande"] for l in donnees_PF}).T
df_dem.columns = [noms_periodes[t] for t in periodes]
df_dem.index.name = "Référence"
display(df_dem)


Produits Finis :


,prix,stock_init,ct,MLS,sigma_D,C_lancement,charge
Référence,,,,,,,
PF1,800.0,300.0,1.0,50.0,40.0,700.0,4.0
PF2,1200.0,150.0,1.0,30.0,25.0,900.0,6.0



Demande client par période :


,Juin,Juillet,Août,Septembre
Référence,,,,
PF1,200,220,210,230
PF2,100,110,105,120


### 2.4 Nomenclature BOM (Bill of Materials)

- `a[i][j]` = quantité de MP $i$ nécessaire pour produire 1 unité de SF $j$
- `b[j][l]` = quantité de SF $j$ nécessaire pour produire 1 unité de PF $l$


In [6]:
# a[i][j] : MP → SF
a = {
    "MP1": {"SF1": 2, "SF2": 1, "SF3": 0},
    "MP2": {"SF1": 0, "SF2": 3, "SF3": 1},
    "MP3": {"SF1": 1, "SF2": 0, "SF3": 2},
    "MP4": {"SF1": 3, "SF2": 2, "SF3": 0},
    "MP5": {"SF1": 0, "SF2": 0, "SF3": 1},
}

# b[j][l] : SF → PF
b = {
    "SF1": {"PF1": 2, "PF2": 1},
    "SF2": {"PF1": 1, "PF2": 2},
    "SF3": {"PF1": 0, "PF2": 1},
}

print("BOM — MP → SF (a[i][j]) :")
display(pd.DataFrame(a).T.fillna(0).astype(int))

print("\nBOM — SF → PF (b[j][l]) :")
display(pd.DataFrame(b).T.fillna(0).astype(int))


BOM — MP → SF (a[i][j]) :


,SF1,SF2,SF3
MP1,2,1,0
MP2,0,3,1
MP3,1,0,2
MP4,3,2,0
MP5,0,0,1



BOM — SF → PF (b[j][l]) :


,PF1,PF2
SF1,2,1
SF2,1,2
SF3,0,1


### 2.5 Pipeline déjà engagé (données figées — extraites de SAP)

Ces données représentent les flux **déjà décidés** avant le démarrage du modèle :
- `OC[i][t]` : commandes fournisseur déjà passées, arrivant en période $t$
- `WIP_SF[j][t]` : encours de production SF déjà lancé, sortant en $t$
- `WIP_PF[l][t]` : encours de production PF déjà lancé, sortant en $t$

> Ces valeurs sont des **paramètres** (pas des variables) : le modèle ne peut pas les modifier.


In [7]:
# Commandes fournisseur déjà passées (en transit)
OC = {
    "MP1": {1: 1000, 2: 500,  3: 0, 4: 0},
    "MP2": {1: 2000, 2: 1000, 3: 0, 4: 0},
    "MP3": {1: 300,  2: 200,  3: 0, 4: 0},
    "MP4": {1: 3000, 2: 1500, 3: 0, 4: 0},
    "MP5": {1: 100,  2: 0,    3: 0, 4: 0},
}

# Encours de production SF déjà lancés
WIP_SF = {
    "SF1": {1: 200, 2: 0, 3: 0, 4: 0},
    "SF2": {1: 300, 2: 0, 3: 0, 4: 0},
    "SF3": {1: 50,  2: 0, 3: 0, 4: 0},
}

# Encours de production PF déjà lancés
WIP_PF = {
    "PF1": {1: 100, 2: 0, 3: 0, 4: 0},
    "PF2": {1: 50,  2: 0, 3: 0, 4: 0},
}

print("✅ Pipeline SAP chargé")
print("   OC  (commandes MP en transit)   :", {i: sum(OC[i].values()) for i in OC}, "unités totales")
print("   WIP SF (encours production SF)  :", {j: sum(WIP_SF[j].values()) for j in WIP_SF})
print("   WIP PF (encours production PF)  :", {l: sum(WIP_PF[l].values()) for l in WIP_PF})


✅ Pipeline SAP chargé
   OC  (commandes MP en transit)   : {'MP1': 1500, 'MP2': 3000, 'MP3': 500, 'MP4': 4500, 'MP5': 100} unités totales
   WIP SF (encours production SF)  : {'SF1': 200, 'SF2': 300, 'SF3': 50}
   WIP PF (encours production PF)  : {'PF1': 100, 'PF2': 50}


### 2.6 Capacités et volumes

- **Volumes** de stockage unitaires (m³/unité)
- **Capacité globale** de stockage (m³)
- **Capacité de production** (heures/mois, réduite en août pour congés)


In [8]:
# Volumes unitaires de stockage (m³/unité)
volume_MP = {"MP1": 0.01, "MP2": 0.02, "MP3": 0.005, "MP4": 0.03, "MP5": 0.008}
volume_SF = {"SF1": 0.05, "SF2": 0.04, "SF3": 0.06}
volume_PF = {"PF1": 0.10, "PF2": 0.15}

# Capacité de stockage globale (m³)
V_max = 5000

# Capacité de production disponible par période (heures)
# → Août = congés fournisseurs → capacité réduite
Cap_SF = {1: 2000, 2: 2000, 3: 800,  4: 2000}
Cap_PF = {1: 1500, 2: 1500, 3: 600,  4: 1500}

print("✅ Capacités définies")
df_cap = pd.DataFrame({
    "Cap_SF (h)": Cap_SF,
    "Cap_PF (h)": Cap_PF
})
df_cap.index = [noms_periodes[t] for t in periodes]
df_cap.index.name = "Période"
display(df_cap)
print(f"   Capacité stockage globale : {V_max} m³")


✅ Capacités définies


,Cap_SF (h),Cap_PF (h)
Période,,
Juin,2000,1500
Juillet,2000,1500
Août,800,600
Septembre,2000,1500


   Capacité stockage globale : 5000 m³


## 3. Calibration EOQ — Calcul des stocks de sécurité

La formule EOQ donne la quantité économique de commande pour chaque référence :

$$Q_k^* = \sqrt{\frac{2 D_k C_k}{h_k}}, \quad h_k = \tau \cdot p_k$$

Le stock de sécurité est calculé à partir du niveau de service cible $\alpha$ = 95% :

$$SS_k = z_{\alpha} \cdot \sigma_{D_k} \cdot \sqrt{LT_k \text{ ou } ct_k}$$

Ces valeurs de $SS_k$ sont injectées dans la contrainte **C6** (taux de service minimum PF).


In [9]:
SS = {}   # Dictionnaire des stocks de sécurité par référence

# Demandes annuelles approchées (pour calcul EOQ)
D_annuel_MP = {"MP1": 6000, "MP2": 12000, "MP3": 2400, "MP4": 20000, "MP5": 800}
D_annuel_SF = {"SF1": 2400, "SF2": 3600,  "SF3": 900}

resultats_eoq = []

# --- MP ---
for i, params in donnees_MP.items():
    h_i    = tau * params["prix"]
    Q_star = np.sqrt(2 * D_annuel_MP[i] * params["C_passation"] / h_i)
    ss_i   = z_alpha * params["sigma_D"] * np.sqrt(params["LT"])
    SS[i]  = ss_i
    resultats_eoq.append({"Référence": i, "Étage": "MP",
                           "EOQ (unités)": round(Q_star), "SS (unités)": round(ss_i),
                           "h ($/an)": round(h_i, 2)})

# --- SF ---
for j, params in donnees_SF.items():
    h_j    = tau * params["prix"]
    Q_star = np.sqrt(2 * D_annuel_SF[j] * params["C_lancement"] / h_j)
    ss_j   = z_alpha * params["sigma_D"] * np.sqrt(params["ct"])
    SS[j]  = ss_j
    resultats_eoq.append({"Référence": j, "Étage": "SF",
                           "EOQ (unités)": round(Q_star), "SS (unités)": round(ss_j),
                           "h ($/an)": round(h_j, 2)})

# --- PF ---
for l, params in donnees_PF.items():
    h_l       = tau * params["prix"]
    D_an      = sum(params["demande"].values()) * 3
    Q_star    = np.sqrt(2 * D_an * params["C_lancement"] / h_l)
    ss_l      = z_alpha * params["sigma_D"] * np.sqrt(params["ct"])
    SS[l]     = ss_l
    resultats_eoq.append({"Référence": l, "Étage": "PF",
                           "EOQ (unités)": round(Q_star), "SS (unités)": round(ss_l),
                           "h ($/an)": round(h_l, 2)})

df_eoq = pd.DataFrame(resultats_eoq).set_index("Référence")
print("📊 Résultats EOQ :")
display(df_eoq)


📊 Résultats EOQ :


,Étage,EOQ (unités),SS (unités),h ($/an)
Référence,,,,
MP1,MP,600,467,10.0
MP2,MP,894,248,6.0
MP3,MP,316,187,24.0
MP4,MP,1155,495,3.0
MP5,MP,179,143,40.0
SF1,SF,179,99,60.0
SF2,SF,245,148,36.0
SF3,SF,110,70,90.0
PF1,PF,150,66,160.0


## 4. Modélisation MILP (PuLP)

### 4.1 Initialisation du modèle


In [10]:
modele = pulp.LpProblem("Optimisation_Stock_Multi_Etages", pulp.LpMinimize)

print("✅ Modèle MILP initialisé :", modele.name)


✅ Modèle MILP initialisé : Optimisation_Stock_Multi_Etages


### 4.2 Variables de décision

| Variable | Type | Signification |
|----------|------|---------------|
| $S_{i,t}^{MP}$, $S_{j,t}^{SF}$, $S_{l,t}^{PF}$ | Continue ≥ 0 | Stock en **fin** de période $t$ |
| $O_{i,t}^{MP}$ | Continue ≥ 0 | Quantité commandée, lancée en **début** de $t$ |
| $P_{j,t}^{SF}$, $P_{l,t}^{PF}$ | Continue ≥ 0 | Quantité produite, lancée en **début** de $t$ |
| $x_{i,t}$ | Binaire | Commande MP passée (1) ou non (0) en $t$ |
| $y_{j,t}^{SF}$, $y_{l,t}^{PF}$ | Binaire | OF lancé (1) ou non (0) en $t$ |


In [11]:
# --- Stocks (fin de période t) ---
S_MP = pulp.LpVariable.dicts("S_MP",
       [(i, t) for i in donnees_MP for t in periodes], lowBound=0, cat="Continuous")

S_SF = pulp.LpVariable.dicts("S_SF",
       [(j, t) for j in donnees_SF for t in periodes], lowBound=0, cat="Continuous")

S_PF = pulp.LpVariable.dicts("S_PF",
       [(l, t) for l in donnees_PF for t in periodes], lowBound=0, cat="Continuous")

# --- Quantités commandées (MP) et produites (SF, PF) ---
O_MP = pulp.LpVariable.dicts("O_MP",
       [(i, t) for i in donnees_MP for t in periodes], lowBound=0, cat="Continuous")

P_SF = pulp.LpVariable.dicts("P_SF",
       [(j, t) for j in donnees_SF for t in periodes], lowBound=0, cat="Continuous")

P_PF = pulp.LpVariable.dicts("P_PF",
       [(l, t) for l in donnees_PF for t in periodes], lowBound=0, cat="Continuous")

# --- Variables binaires ---
x    = pulp.LpVariable.dicts("x",    [(i, t) for i in donnees_MP for t in periodes], cat="Binary")
y_SF = pulp.LpVariable.dicts("y_SF", [(j, t) for j in donnees_SF for t in periodes], cat="Binary")
y_PF = pulp.LpVariable.dicts("y_PF", [(l, t) for l in donnees_PF for t in periodes], cat="Binary")

# --- Résumé ---
nb_cont = len(S_MP) + len(S_SF) + len(S_PF) + len(O_MP) + len(P_SF) + len(P_PF)
nb_bin  = len(x) + len(y_SF) + len(y_PF)
print(f"✅ Variables créées")
print(f"   Continues : {nb_cont}")
print(f"   Binaires  : {nb_bin}")


✅ Variables créées
   Continues : 80
   Binaires  : 40


### 4.3 Fonction objectif

$$\min Z = \sum_{t=1}^{T}\left( \sum_i p_i^{MP} S_{i,t}^{MP} + \sum_j p_j^{SF} S_{j,t}^{SF} + \sum_l p_l^{PF} S_{l,t}^{PF} \right)$$

Minimiser la **valeur totale du stock cumulée** sur tout l'horizon (juin → septembre).


In [12]:
modele += (
    pulp.lpSum(donnees_MP[i]["prix"] * S_MP[(i, t)] for i in donnees_MP for t in periodes)
  + pulp.lpSum(donnees_SF[j]["prix"] * S_SF[(j, t)] for j in donnees_SF for t in periodes)
  + pulp.lpSum(donnees_PF[l]["prix"] * S_PF[(l, t)] for l in donnees_PF for t in periodes),
    "Minimiser_Valeur_Totale_Stock"
)

print("✅ Fonction objectif définie")


✅ Fonction objectif définie


### 4.4 Contraintes

#### C1 — Bilan de stock Matière Première

$$S_{i,t}^{MP} = S_{i,t-1}^{MP} + OC_{i,t}^{MP} + O_{i,\,t-LT_i}^{MP} - \sum_j a_{ij}\, P_{j,t}^{SF} \quad \forall i, t$$

- $OC_{i,t}^{MP}$ = commandes **déjà passées**, arrivant en $t$ (figées, extraites de SAP)
- $O_{i,t-LT_i}^{MP}$ = commandes **décidées** par le modèle, lancées en $t - LT_i$
- Si $t - LT_i < 1$ : la commande n'est pas encore décidée dans le modèle → terme nul


In [13]:
for i in donnees_MP:
    LT_i = donnees_MP[i]["LT"]
    for t in periodes:
        S_prev = donnees_MP[i]["stock_init"] if t == 1 else S_MP[(i, t-1)]

        t_commande = t - LT_i
        reception_nouvelle = O_MP[(i, t_commande)] if t_commande >= 1 else 0

        conso_MP = pulp.lpSum(a[i][j] * P_SF[(j, t)]
                              for j in donnees_SF if a[i][j] > 0)

        modele += (
            S_MP[(i, t)] == S_prev + OC[i][t] + reception_nouvelle - conso_MP,
            f"C1_Bilan_MP_{i}_t{t}"
        )

print("✅ C1 — Bilans de stock MP ajoutés")


✅ C1 — Bilans de stock MP ajoutés


#### C2 — Bilan de stock Semi-Fini

$$S_{j,t}^{SF} = S_{j,t-1}^{SF} + WIP_{j,t}^{SF} + P_{j,\,t-ct_j}^{SF} - \sum_l b_{jl}\, P_{l,t}^{PF} \quad \forall j, t$$

- $WIP_{j,t}^{SF}$ = encours production SF **déjà lancé**, sortant en $t$ (figé SAP)
- $P_{j,t-ct_j}^{SF}$ = OF **décidé** par le modèle, lancé en $t - ct_j$, sortant en $t$
- Si $t - ct_j < 1$ : pas encore de sortie production modèle → terme nul


In [14]:
for j in donnees_SF:
    ct_j = donnees_SF[j]["ct"]
    for t in periodes:
        S_prev = donnees_SF[j]["stock_init"] if t == 1 else S_SF[(j, t-1)]

        t_lancement = t - ct_j
        reception_SF = P_SF[(j, t_lancement)] if t_lancement >= 1 else 0

        conso_SF = pulp.lpSum(b[j][l] * P_PF[(l, t)]
                              for l in donnees_PF if b[j][l] > 0)

        modele += (
            S_SF[(j, t)] == S_prev + WIP_SF[j][t] + reception_SF - conso_SF,
            f"C2_Bilan_SF_{j}_t{t}"
        )

print("✅ C2 — Bilans de stock SF ajoutés")


✅ C2 — Bilans de stock SF ajoutés


#### C3 — Bilan de stock Produit Fini

$$S_{l,t}^{PF} = S_{l,t-1}^{PF} + WIP_{l,t}^{PF} + P_{l,\,t-ct_l}^{PF} - D_{l,t} \quad \forall l, t$$

Seul étage avec une **demande externe réelle** $D_{l,t}$ (demande client).


In [15]:
for l in donnees_PF:
    ct_l = donnees_PF[l]["ct"]
    for t in periodes:
        S_prev = donnees_PF[l]["stock_init"] if t == 1 else S_PF[(l, t-1)]

        t_lancement = t - ct_l
        reception_PF = P_PF[(l, t_lancement)] if t_lancement >= 1 else 0

        demande_t = donnees_PF[l]["demande"][t]

        modele += (
            S_PF[(l, t)] == S_prev + WIP_PF[l][t] + reception_PF - demande_t,
            f"C3_Bilan_PF_{l}_t{t}"
        )

print("✅ C3 — Bilans de stock PF ajoutés")


✅ C3 — Bilans de stock PF ajoutés


#### C4 — Initialisation des stocks (intégrée dans C1/C2/C3)

Les stocks initiaux $S_{i,0}^{MP}$, $S_{j,0}^{SF}$, $S_{l,0}^{PF}$ sont directement utilisés dans C1/C2/C3 via la condition `t == 1` → pas de contrainte séparée nécessaire.

---

#### C5 — Cible de valeur de stock fin d'année fiscale

$$\sum_i p_i^{MP} S_{i,T}^{MP} + \sum_j p_j^{SF} S_{j,T}^{SF} + \sum_l p_l^{PF} S_{l,T}^{PF} \leq Target$$

C'est la **contrainte principale** du modèle : la valeur totale du stock en fin de $T$ (septembre) doit être ≤ 10 000 000 $.


In [16]:
modele += (
    pulp.lpSum(donnees_MP[i]["prix"] * S_MP[(i, T)] for i in donnees_MP)
  + pulp.lpSum(donnees_SF[j]["prix"] * S_SF[(j, T)] for j in donnees_SF)
  + pulp.lpSum(donnees_PF[l]["prix"] * S_PF[(l, T)] for l in donnees_PF)
  <= TARGET,
    "C5_Cible_Fiscale_Fin_Septembre"
)

print("✅ C4 — Initialisation intégrée dans C1/C2/C3")
print(f"✅ C5 — Cible fiscale {TARGET:,.0f} $ ajoutée")


✅ C4 — Initialisation intégrée dans C1/C2/C3
✅ C5 — Cible fiscale 10,000,000 $ ajoutée


#### C6 — Taux de service minimum (Produit Fini)

$$S_{l,t}^{PF} \geq SS_l \quad \forall l, t$$

Le stock de PF ne doit jamais descendre sous le stock de sécurité $SS_l$ calculé via EOQ.


In [17]:
for l in donnees_PF:
    for t in periodes:
        modele += (
            S_PF[(l, t)] >= SS[l],
            f"C6_TauxService_{l}_t{t}"
        )

print("✅ C6 — Taux de service minimum PF ajouté")
print("   Stocks de sécurité :")
for l in donnees_PF:
    print(f"   {l} : SS = {SS[l]:.0f} unités")


✅ C6 — Taux de service minimum PF ajouté
   Stocks de sécurité :
   PF1 : SS = 66 unités
   PF2 : SS = 41 unités


#### C7 — Capacité de production

$$\sum_j c_j \cdot P_{j,t}^{SF} \leq Cap_t^{SF}, \qquad \sum_l c_l \cdot P_{l,t}^{PF} \leq Cap_t^{PF} \quad \forall t$$

- $c_j, c_l$ = **charge unitaire** (heures nécessaires pour produire 1 unité)
- $Cap_t$ = **capacité disponible** (heures disponibles sur la période $t$)

La charge totale générée doit rester inférieure à la capacité disponible (réduite en août).


In [18]:
for t in periodes:
    modele += (
        pulp.lpSum(donnees_SF[j]["charge"] * P_SF[(j, t)] for j in donnees_SF) <= Cap_SF[t],
        f"C7_Capacite_SF_t{t}"
    )
    modele += (
        pulp.lpSum(donnees_PF[l]["charge"] * P_PF[(l, t)] for l in donnees_PF) <= Cap_PF[t],
        f"C7_Capacite_PF_t{t}"
    )

print("✅ C7 — Capacité de production ajoutée")


✅ C7 — Capacité de production ajoutée


#### C8 — Capacité de stockage globale (volume)

$$\sum_i v_i^{MP} S_{i,t}^{MP} + \sum_j v_j^{SF} S_{j,t}^{SF} + \sum_l v_l^{PF} S_{l,t}^{PF} \leq V_{max} \quad \forall t$$


In [19]:
for t in periodes:
    modele += (
        pulp.lpSum(volume_MP[i] * S_MP[(i, t)] for i in donnees_MP)
      + pulp.lpSum(volume_SF[j] * S_SF[(j, t)] for j in donnees_SF)
      + pulp.lpSum(volume_PF[l] * S_PF[(l, t)] for l in donnees_PF)
      <= V_max,
        f"C8_Capacite_Stockage_t{t}"
    )

print("✅ C8 — Capacité de stockage globale ajoutée")


✅ C8 — Capacité de stockage globale ajoutée


#### C9 — MOQ et liaison binaire — Achat MP

$$O_{i,t}^{MP} \geq MOQ_i \cdot x_{i,t}, \qquad O_{i,t}^{MP} \leq M \cdot x_{i,t} \quad \forall i, t$$

- Si $x_{i,t} = 0$ → $O_{i,t} = 0$ (pas de commande)
- Si $x_{i,t} = 1$ → $O_{i,t} \geq MOQ_i$ (commande au minimum égale à la MOQ)


In [20]:
for i in donnees_MP:
    MOQ_i = donnees_MP[i]["MOQ"]
    for t in periodes:
        modele += (O_MP[(i, t)] >= MOQ_i * x[(i, t)],  f"C9a_MOQ_min_{i}_t{t}")
        modele += (O_MP[(i, t)] <= M     * x[(i, t)],  f"C9b_MOQ_max_{i}_t{t}")

print("✅ C9 — MOQ et binaire achat MP ajoutés")


✅ C9 — MOQ et binaire achat MP ajoutés


#### C10 — MLS et liaison binaire — Production SF et PF

$$P_{j,t}^{SF} \geq MLS_j \cdot y_{j,t}^{SF}, \qquad P_{j,t}^{SF} \leq M \cdot y_{j,t}^{SF} \quad \forall j, t$$
$$P_{l,t}^{PF} \geq MLS_l \cdot y_{l,t}^{PF}, \qquad P_{l,t}^{PF} \leq M \cdot y_{l,t}^{PF} \quad \forall l, t$$

- Si $y = 0$ → pas d'OF lancé → $P = 0$
- Si $y = 1$ → OF lancé → $P \geq MLS$ (taille de lot minimale)


In [21]:
for j in donnees_SF:
    MLS_j = donnees_SF[j]["MLS"]
    for t in periodes:
        modele += (P_SF[(j, t)] >= MLS_j * y_SF[(j, t)], f"C10a_MLS_SF_min_{j}_t{t}")
        modele += (P_SF[(j, t)] <= M     * y_SF[(j, t)], f"C10b_MLS_SF_max_{j}_t{t}")

for l in donnees_PF:
    MLS_l = donnees_PF[l]["MLS"]
    for t in periodes:
        modele += (P_PF[(l, t)] >= MLS_l * y_PF[(l, t)], f"C10a_MLS_PF_min_{l}_t{t}")
        modele += (P_PF[(l, t)] <= M     * y_PF[(l, t)], f"C10b_MLS_PF_max_{l}_t{t}")

print("✅ C10 — MLS et binaires production SF/PF ajoutés")


✅ C10 — MLS et binaires production SF/PF ajoutés

#### C11 — Fermeture fournisseur en Août (aucune commande possible)

$$O_{i,t}^{MP} = 0 \quad \forall i, \; t \in t_{août}$$

---

#### C12 — Couverture de stock avant fermeture (anticipation)

$$S_{i,\, t_{juillet}}^{MP} \geq \sum_{t \in t_{août}} \sum_j a_{ij}\, P_{j,t}^{SF} \quad \forall i$$

Le stock MP fin juillet doit couvrir **toute la consommation prévue en août**, puisqu'aucun réapprovisionnement ne sera possible.  
→ Le modèle va donc naturellement créer un **pic de stock en juillet** pour sécuriser août.


In [22]:
# C11 — Pas de commande MP en août
for i in donnees_MP:
    for t in t_aout:
        modele += (O_MP[(i, t)] == 0, f"C11_Fermeture_Fournisseur_{i}_t{t}")

print("✅ C11 — Fermeture fournisseur août ajoutée")

# C12 — Stock MP fin juillet >= consommation prévue en août
for i in donnees_MP:
    conso_aout = pulp.lpSum(
        a[i][j] * P_SF[(j, t)]
        for t in t_aout
        for j in donnees_SF
        if a[i][j] > 0
    )
    modele += (
        S_MP[(i, t_juillet)] >= conso_aout,
        f"C12_Couverture_Aout_{i}"
    )

print("✅ C12 — Couverture stock avant fermeture ajoutée")


✅ C11 — Fermeture fournisseur août ajoutée
✅ C12 — Couverture stock avant fermeture ajoutée


#### C13 — Non-négativité et domaine des variables

$$S^{MP}, S^{SF}, S^{PF}, O^{MP}, P^{SF}, P^{PF} \geq 0, \qquad x_{i,t},\, y_{j,t},\, y_{l,t} \in \{0,1\}$$

→ Déjà intégré directement dans la déclaration des variables via `lowBound=0` et `cat="Binary"`.


In [23]:
print("✅ C13 — Non-négativité et domaine binaire intégrés à la déclaration des variables")

# ============================================================
# RÉSUMÉ FINAL DU MODÈLE
# ============================================================
nb_vars_cont = len(S_MP) + len(S_SF) + len(S_PF) + len(O_MP) + len(P_SF) + len(P_PF)
nb_vars_bin  = len(x) + len(y_SF) + len(y_PF)
nb_cst       = len(modele.constraints)

print()
print("=" * 55)
print("  RÉSUMÉ DU MODÈLE")
print("=" * 55)
print(f"  Type de problème      : MILP")
print(f"  Variables continues   : {nb_vars_cont}")
print(f"  Variables binaires    : {nb_vars_bin}")
print(f"  Nombre de contraintes : {nb_cst}")
print(f"  Horizon               : {len(periodes)} mois ({', '.join(noms_periodes.values())})")
print(f"  Cible fiscale         : {TARGET:,.0f} $")
print(f"  Solveur prévu         : CBC (via PuLP, open-source)")
print("=" * 55)
print()
print("✅ Modélisation complète.")
print("   Passez au notebook suivant pour la RÉSOLUTION.")


✅ C13 — Non-négativité et domaine binaire intégrés à la déclaration des variables

  RÉSUMÉ DU MODÈLE
  Type de problème      : MILP
  Variables continues   : 80
  Variables binaires    : 40
  Nombre de contraintes : 151
  Horizon               : 4 mois (Juin, Juillet, Août, Septembre)
  Cible fiscale         : 10,000,000 $
  Solveur prévu         : CBC (via PuLP, open-source)

✅ Modélisation complète.
   Passez au notebook suivant pour la RÉSOLUTION.


---
## 5. Résolution

Comme dans la structure AMPL/GLPK (`solve;` puis `display ...;`), on résout ici le modèle MILP
avec le solveur CBC (fourni avec PuLP), puis on affiche et exploite les résultats.


### 5.1 Appel du solveur (`solve;`)

In [ ]:
# ---------- Résolution ----------
solveur = pulp.PULP_CBC_CMD(msg=1)   # msg=1 pour voir le log CBC, msg=0 pour le masquer
statut = modele.solve(solveur)

print("=" * 55)
print("  STATUT DE LA RÉSOLUTION")
print("=" * 55)
print(f"  Statut       : {pulp.LpStatus[modele.status]}")
print(f"  Valeur objectif (Z) : {pulp.value(modele.objective):,.2f} $")
print("=" * 55)


### 5.2 Vérification de la cible fiscale (C5)

In [ ]:
# ---------- Affichage ----------
valeur_fin_sept = (
    sum(donnees_MP[i]["prix"] * S_MP[(i, T)].value() for i in donnees_MP)
  + sum(donnees_SF[j]["prix"] * S_SF[(j, T)].value() for j in donnees_SF)
  + sum(donnees_PF[l]["prix"] * S_PF[(l, T)].value() for l in donnees_PF)
)

print(f"Valeur de stock fin {noms_periodes[T]} : {valeur_fin_sept:,.0f} $")
print(f"Cible fiscale (TARGET)              : {TARGET:,.0f} $")
ecart = TARGET - valeur_fin_sept
print(f"Marge par rapport à la cible         : {ecart:,.0f} $ "
      f"({'✅ cible respectée' if valeur_fin_sept <= TARGET else '❌ cible dépassée'})")


### 5.3 Extraction des résultats — Stocks $S_{i,t}$

On construit un DataFrame par étage (MP / SF / PF) avec le stock en fin de chaque période.


In [ ]:
def extraire_stock(var_dict, donnees, label):
    lignes = []
    for ref in donnees:
        for t in periodes:
            lignes.append({
                "Référence": ref,
                "Période": noms_periodes[t],
                "t": t,
                "Stock (unités)": var_dict[(ref, t)].value(),
                "Valeur ($)": var_dict[(ref, t)].value() * donnees[ref]["prix"],
            })
    df = pd.DataFrame(lignes)
    print(f"Stocks {label} :")
    display(df.pivot(index="Référence", columns="Période", values="Stock (unités)")[
        [noms_periodes[t] for t in periodes]
    ])
    return df

df_stock_MP = extraire_stock(S_MP, donnees_MP, "MP")
df_stock_SF = extraire_stock(S_SF, donnees_SF, "SF")
df_stock_PF = extraire_stock(S_PF, donnees_PF, "PF")


### 5.4 Extraction des résultats — Commandes $O^{MP}$ et Productions $P^{SF}, P^{PF}$

In [ ]:
def extraire_flux(var_dict, donnees, label):
    lignes = []
    for ref in donnees:
        for t in periodes:
            lignes.append({
                "Référence": ref,
                "Période": noms_periodes[t],
                "t": t,
                f"{label} (unités)": var_dict[(ref, t)].value(),
            })
    df = pd.DataFrame(lignes)
    print(f"{label} :")
    display(df.pivot(index="Référence", columns="Période", values=f"{label} (unités)")[
        [noms_periodes[t] for t in periodes]
    ])
    return df

df_O_MP = extraire_flux(O_MP, donnees_MP, "Commandes O_MP")
df_P_SF = extraire_flux(P_SF, donnees_SF, "Production P_SF")
df_P_PF = extraire_flux(P_PF, donnees_PF, "Production P_PF")


### 5.5 Variables binaires — commandes/lancements déclenchés ($x$, $y^{SF}$, $y^{PF}$)

In [ ]:
def extraire_binaire(var_dict, donnees, label):
    lignes = []
    for ref in donnees:
        for t in periodes:
            lignes.append({
                "Référence": ref,
                "Période": noms_periodes[t],
                f"{label}": int(round(var_dict[(ref, t)].value())),
            })
    df = pd.DataFrame(lignes)
    print(f"{label} :")
    display(df.pivot(index="Référence", columns="Période", values=f"{label}")[
        [noms_periodes[t] for t in periodes]
    ])
    return df

df_x    = extraire_binaire(x,    donnees_MP, "x (commande MP)")
df_y_SF = extraire_binaire(y_SF, donnees_SF, "y_SF (OF lancé)")
df_y_PF = extraire_binaire(y_PF, donnees_PF, "y_PF (OF lancé)")


### 5.6 Valeur totale du stock par période (fonction objectif détaillée)

In [ ]:
valeurs_par_periode = []
for t in periodes:
    v_mp = sum(donnees_MP[i]["prix"] * S_MP[(i, t)].value() for i in donnees_MP)
    v_sf = sum(donnees_SF[j]["prix"] * S_SF[(j, t)].value() for j in donnees_SF)
    v_pf = sum(donnees_PF[l]["prix"] * S_PF[(l, t)].value() for l in donnees_PF)
    valeurs_par_periode.append({
        "Période": noms_periodes[t], "t": t,
        "Valeur MP ($)": v_mp, "Valeur SF ($)": v_sf, "Valeur PF ($)": v_pf,
        "Valeur totale ($)": v_mp + v_sf + v_pf,
    })

df_valeur = pd.DataFrame(valeurs_par_periode).set_index("Période")
display(df_valeur)

print(f"\nValeur totale cumulée (fonction objectif Z) : {pulp.value(modele.objective):,.0f} $")


### 5.7 Graphiques

In [ ]:
import matplotlib.pyplot as plt

# --- Graphique 1 : évolution de la valeur de stock par étage ---
fig, ax = plt.subplots(figsize=(9, 5))
labels_periodes = [noms_periodes[t] for t in periodes]

ax.plot(labels_periodes, df_valeur["Valeur MP ($)"], marker="o", label="MP")
ax.plot(labels_periodes, df_valeur["Valeur SF ($)"], marker="o", label="SF")
ax.plot(labels_periodes, df_valeur["Valeur PF ($)"], marker="o", label="PF")
ax.plot(labels_periodes, df_valeur["Valeur totale ($)"], marker="s", linewidth=2.5,
        color="black", label="Total")
ax.axhline(TARGET, color="red", linestyle="--", label=f"Cible fiscale ({TARGET:,.0f} $)")

ax.set_title("Évolution de la valeur de stock par étage")
ax.set_xlabel("Période")
ax.set_ylabel("Valeur du stock ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- Graphique 2 : valeur de stock empilée (barres) par étage et par période ---
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(labels_periodes, df_valeur["Valeur MP ($)"], label="MP")
ax.bar(labels_periodes, df_valeur["Valeur SF ($)"],
       bottom=df_valeur["Valeur MP ($)"], label="SF")
ax.bar(labels_periodes, df_valeur["Valeur PF ($)"],
       bottom=df_valeur["Valeur MP ($)"] + df_valeur["Valeur SF ($)"], label="PF")
ax.axhline(TARGET, color="red", linestyle="--", label=f"Cible fiscale ({TARGET:,.0f} $)")

ax.set_title("Valeur de stock par étage (empilée)")
ax.set_xlabel("Période")
ax.set_ylabel("Valeur du stock ($)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- Graphique 3 : commandes MP et production SF/PF déclenchées ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)

for ax_, df_, val_col, titre in zip(
        axes,
        [df_O_MP, df_P_SF, df_P_PF],
        ["Commandes O_MP (unités)", "Production P_SF (unités)", "Production P_PF (unités)"],
        ["Commandes MP", "Production SF", "Production PF"]
    ):
    pivot = df_.pivot(index="Période", columns="Référence", values=val_col).loc[labels_periodes]
    pivot.plot(kind="bar", ax=ax_, legend=True)
    ax_.set_title(titre)
    ax_.set_xlabel("Période")
    ax_.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


### 5.8 Résumé final

In [ ]:
print("=" * 60)
print("  RÉSUMÉ DE LA RÉSOLUTION")
print("=" * 60)
print(f"  Statut solveur          : {pulp.LpStatus[modele.status]}")
print(f"  Valeur objectif Z       : {pulp.value(modele.objective):,.0f} $")
print(f"  Valeur stock fin {noms_periodes[T]:<9}: {valeur_fin_sept:,.0f} $")
print(f"  Cible fiscale           : {TARGET:,.0f} $")
print(f"  Cible respectée         : "
      f"{'✅ Oui' if valeur_fin_sept <= TARGET else '❌ Non'}")
print("=" * 60)
